In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from dotenv import load_dotenv
import sqlite3
import requests
import os

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [4]:
search_tool = DuckDuckGoSearchRun(region="us-en")

In [5]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num

        elif operation == "sub":
            result = first_num - second_num

        elif operation == "mul":
            result = first_num * second_num

        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num

        else:
            return {"error": f"Unsupported operation '{operation}'"}

        return {
            "first_num": first_num,
            "second_num": second_num,
            "operation": operation,
            "result": result
        }

    except Exception as e:
        return {"error": str(e)}


In [6]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol
    using Alpha Vantage.
    """

    api_key = os.getenv("ALPHA_VANTAGE_API_KEY")

    url = (
        f"https://www.alphavantage.co/query?"
        f"function=GLOBAL_QUOTE"
        f"&symbol={symbol}"
        f"&apikey={api_key}"
    )

    r = requests.get(url)
    return r.json()

In [7]:
tools = [
    search_tool,
    get_stock_price,
    calculator
]

# Bind tools to Groq LLM
llm_with_tools = llm.bind_tools(tools)

In [8]:
# -------------------
# 3. State
# -------------------

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [9]:
# 4. Nodes
# -------------------

def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""

    messages = state["messages"]

    response = llm_with_tools.invoke(messages)

    return {
        "messages": [response]
    }


tool_node = ToolNode(tools)

In [10]:
# 5. Checkpointer
# -------------------

conn = sqlite3.connect(
    database="chatbot.db",
    check_same_thread=False
)

checkpointer = SqliteSaver(conn=conn)


In [11]:
# 6. Graph
# -------------------

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges(
    "chat_node",
    tools_condition
)

graph.add_edge("tools", "chat_node")

chatbot = graph.compile(
    checkpointer=checkpointer
)


In [12]:
# 7. Helper
# -------------------

def retrieve_all_threads():

    all_threads = set()

    for checkpoint in checkpointer.list(None):
        all_threads.add(
            checkpoint.config["configurable"]["thread_id"]
        )

    return list(all_threads)

In [14]:
from langchain_core.messages import HumanMessage

In [16]:
out = chatbot.invoke(
    {
        "messages": [HumanMessage(content="Hello")]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

print(out["messages"][-1].content)

Hello! How can I help you today?


In [17]:
out = chatbot.invoke(
    {
        "messages": [HumanMessage(content="What is 2 X 3 ?")]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

print(out["messages"][-1].content)

2 × 3 equals **6**.


In [19]:
out = chatbot.invoke(
    {
        "messages": [HumanMessage(content="what is stock price of apple, how much it cost to purchase 50 share ?")]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

print(out["messages"][-1].content)

**Apple Inc. (AAPL) – Latest Price (2026‑09‑04)**  
- Current trading price: **$319.97** per share  

**Cost to buy 50 shares**  
- 50 shares × $319.97/share = **$15,998.50**  

So, purchasing 50 shares of Apple would cost you approximately **$15,998.50** (before any brokerage fees or commissions).
